In [1]:
import os
import time
import pandas as pd
import numpy as np
import plotly.express as px

# Enable future behavior warning enforcement
pd.set_option('future.no_silent_downcasting', True)

def clean_and_parse(df):
    # ---- Select & Rename Columns ----
    numeric_cols = ['Subtotal', 'Shipping', 'Taxes', 'Total',
                'Discount Amount', 'Lineitem quantity', 
                'Lineitem price', 'Device ID', 'Lineitem discount',
                'Tax 1 Value', 'Tax 2 Value']
    # Add a cleaning step after reading or inside clean_and_parse()
    # that ensures numeric fields are really numeric, even if the CSV came in messy:
    for col in numeric_cols:
        if col in master_df.columns:
            master_df[col] = (
                master_df[col]
                .astype(str)  # in case it's mixed
                .str.replace(r'[^0-9\.\-]', '', regex=True)
                .replace('', np.nan)
                .astype(float)
            )
    df = df[
        ['Name', 'Paid at', 'Fulfilled at', 'Subtotal', 'Shipping', 'Taxes', 'Total',
         'Discount Code', 'Discount Amount', 'Lineitem quantity', 'Lineitem name',
         'Lineitem price', 'Lineitem sku', 'Location', 'Device ID', 'Lineitem discount',
         'Tax 1 Name', 'Tax 1 Value', 'Tax 2 Name', 'Tax 2 Value']
    ].rename(columns={
        'Name': 'Order Number',
        'Paid at': 'Paid At',
        'Fulfilled at': 'Fulfilled At',
        'Lineitem quantity': 'Item Qty',
        'Lineitem name': 'Item Name',
        'Lineitem price': 'Item Price',
        'Lineitem sku': 'Item SKU',
        'Lineitem discount': 'Item Discount'
    })
    
    
    # ---- Fill Missing Location ----
    df['Location'] = df.groupby('Order Number')['Location'].transform(lambda x: x.ffill().bfill())

    # Sort 'Item Name' column values in alphabetical order
    df['Item Name'] = df['Item Name'].astype(str)
    sorted_items = sorted(df['Item Name'].unique())

    # Identify all flight related Item Names
    flight_items = [item for item in sorted_items if 'flight' in item.lower()]

    # Identify all glass of wine related Item Names
    glass_items = [item for item in sorted_items if 'glass' in item.lower()]

    # Identify all wine tasting related Item Name
    tasting_items = [item for item in sorted_items if 'tasting' in item.lower()]

    
    # Normalize all target 'Item Names' to lowercase for consistent comparison
    # ---- Categorize Items ----
    flights_set = set([
    'Wine Tastings - Flight / 90 m',
    'Wine Tastings - NJ Flight / 90 m',
    'wine tastings - PA Flight / 90 m',
    ])
    
    glasses_set = set([
        'Glass of Wine - Glass of Wine - $14',
        'Glass of Wine - Glass of Wine - $15',
        'Glass of Wine - Glass of Wine - $8',
        'Glass of Wine - Glass of Wine- $10',
        'Glass of Wine - Glass of Wine- $11',
        'Glass of Wine - Glass of Wine- $12',
        'Glass of Wine - Glass of Wine- $9',
        'Glass of Wine - Glass of Wine-$8',
        'Glass of Wine - Glass of Wine-7'
    ])    

    tastings_set = set([
        '$10 Reserve Tasting',
        '$5 Tasting',
        '8+ group tasting fee',
        'Additional tastings',
        'Comp Tasting',
        'Group Tasting',
        'Group Tasting Fee',
        'Tasting',
        'Wine Tasting - 90 m',
        'Wine Tasting - 90 minutes',
        'Wine Tastings - BB Tasting / 90m',
        'Wine Tastings - Group Tasting / 90 m',
        'Wine Tastings - PA Tasting',
        'Wine Tastings - PA Tasting / 90 m',
        'Wine Tastings - Tasting $10',
        'Wine Tastings - Tasting / 90 m',
        ])

    # Handles non-string items by converting everything to str first
    # Useful if any set contained non-str types
    flights_set = set(map(lambda x: str(x).strip().lower(), flights_set))
    glasses_set = set(map(lambda x: str(x).strip().lower(), glasses_set))
    tastings_set = set(map(lambda x: str(x).strip().lower(), tastings_set))


    # Normalize 'Item Name' using .strip().lower() and check if it's in one of the sets
    def categorize_item(name):
        name_norm = str(name).strip().lower()
        if name_norm in flights_set:
            return 'Flight'
        elif name_norm in glasses_set:
            return 'Glass'
        elif name_norm in tastings_set:
            return 'Tasting'
        else:
            return np.nan

    df['Item Category'] = df['Item Name'].apply(categorize_item)

    # Redefine the DataFrame with the added 'Item Category' column
    df = df[[
        'Order Number', 'Paid At', 'Fulfilled At', 'Subtotal', 'Shipping', 'Taxes', 'Total', 'Discount Code',
        'Discount Amount', 'Item Qty', 'Item Name', 'Item Category', 'Item Price', 'Item SKU', 'Location',
        'Device ID', 'Item Discount', 'Tax 1 Name', 'Tax 1 Value', 'Tax 2 Name', 'Tax 2 Value'
        ]]

    # ---- Tax-based Location Inference ----
    tax_to_location_map = {
        'NJ State Tax 6.625%': 'Unionville Vineyards',
        'NJ State Tax 6.63%': 'Unionville Vineyards',
        'New Jersey State Tax 6.625%': 'Unionville Vineyards',
        'PA State Tax 6%': 'Ferry Market',
        'Pennsylvania State Tax 6%': 'Ferry Market'
    }

    # For Locations still blank, we can gather where the sale occurred based on the sales tax collected
    # Create filter masks (boolean values of True or False aligned with the rows in our DataFrame,
    # used to identify which rows meet a certain criteria)
    
    mask_glass = df['Location'].isna() & df['Item Name'].str.lower().str.strip().isin(glasses_set)
    mask_tasting = df['Location'].isna() & df['Item Name'].str.lower().str.strip().isin(tastings_set)
    
    # Below, we're saying: For all row locations where a condition (mask) is True, set the 'Location' value
    # using a mapping based on the value we see in 'Tax 1 Name' within these same exact rows (where same condition (mask) is True.
    # Finally, for all row locations matching this condition where 'Tax 1 Name' was blank (NaN),
    # fill its 'Location' cell value with 'Unionville Vineyards'    
    
    # Apply the mapping for wine glasses rows based on 'Tax 1 Name'
    df.loc[mask_glass, 'Location'] = df.loc[mask_glass, 'Tax 1 Name'].map(tax_to_location_map).fillna('Unionville Vineyards')
    # Apply the mapping for wine tastings rows based on 'Tax 1 Name'
    df.loc[mask_tasting, 'Location'] = df.loc[mask_tasting, 'Tax 1 Name'].map(tax_to_location_map).fillna('Unionville Vineyards')


    # Reassign 'Tasting Room' Location to be 'Unionville Vineyards'
    # ---- Normalize Location Names ----
    df.loc[df['Location'].str.lower() == 'tasting room', 'Location'] = 'Unionville Vineyards'
    
    # Redefine df to be only rows with 'Ferry Market' and 'Unionville Vineyards' locations
    
    df = df[df['Location'].isin(['Ferry Market', 'Unionville Vineyards'])].copy()

    # Begin Datetime Code Transformation

    # Step 1: Replace blank or missing 'Paid At' with 'Fulfilled At' (only where 'Paid At' is missing)
    df['Paid At'] = df['Paid At'].fillna(df['Fulfilled At'])
    
    # Step 2: Forward-fill and back-fill 'Paid At' within each 'Order Number' group
    df['Paid At'] = df.groupby('Order Number')['Paid At'].transform(lambda x: x.ffill().bfill())

    # Fill missing 'Paid At' cell values based on the date of an Order Number date before/after it
    filled_dates = df['Paid At'].ffill().bfill()
    df['Paid At'] = np.where(
        df['Paid At'].isna(),
        filled_dates,
        df['Paid At']
    )

    ###### 'Paid At' column contains multiple datetime format strings, requiring handling formats conditionally:
    ###### 1. Detect the format of each row
    ###### 2. Parse them accordingly
    ###### 3. Combine results into one cleaned datetime column
    
    # Identify and normalize all 'Paid At' datetime strings with string format '%Y-%m-%d %H:%M:%S %z'
    # Some datetime strings are of format '%m/%d/%y %H:%M' where others are already '%Y-%m-%d %H:%M:%S %z'

    # Step 1:
    # Make a working copy
    col = df['Paid At'].astype(str)
    
    # Create masks for each format, identifying which rows match which format
    mask_mdy = col.str.match(r'^\d{1,2}/\d{1,2}/\d{2} \d{1,2}:\d{2}$') # Order Number #138208 '6/30/25 16:32'
    mask_ymd_tz = col.str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} [+-]\d{4}$') # Order Number #59033 '2020-02-09 18:35:33 -0500'
    
    # Step 2:
    # Parse short format ('6/30/25 16:32'), assign a time zone (localize), convert to UTC
    parsed_short = pd.to_datetime(
        col[mask_mdy],
        format='%m/%d/%y %H:%M',
        errors='coerce'
    ).dt.tz_localize('America/New_York').dt.tz_convert('UTC')

    # Parse long format directly (already has timezone info)
    parsed_long = pd.to_datetime(
        col[mask_ymd_tz],
        errors='coerce',
        utc=True)

    # Create an empty results container (an empty pandas Series)
    # This creates an empty column with the same index as the original, and correct datatype (UTC datetime)
    combined = pd.Series(index=col.index, dtype='datetime64[ns, UTC]')
    
    # Step 3: Combine results
    # Put each parsed result back into the right rows
    # Replace original column with the cleaned version
    combined.loc[mask_mdy] = parsed_short
    combined.loc[mask_ymd_tz] = parsed_long
    df['Paid At'] = combined

    # ---- Feature Engineering ----
    # Split 'Paid At' column into Year, Date, and Time columns
    df['Year'] = df['Paid At'].dt.year.astype('Int64')
    df['Date'] = df['Paid At'].dt.date
    df['Time'] = df['Paid At'].dt.time

    # ---- Reorder Columns ----
    # Redefine df to include Year, Date, and Time columns
    df = df[['Order Number', 'Paid At', 'Fulfilled At', 'Year', 'Date', 'Time',
             'Subtotal', 'Shipping', 'Taxes', 'Total', 'Discount Code', 'Discount Amount',
             'Item Qty', 'Item Name', 'Item Category', 'Item Price', 'Item SKU',
             'Location', 'Device ID', 'Item Discount', 'Tax 1 Name', 'Tax 1 Value',
             'Tax 2 Name', 'Tax 2 Value']]

    return df

In [2]:
def ytd_sales_analysis_pipeline(df, debug=True):
    """
    Execute a multi-year Year-to-Date (YTD) sales analysis pipeline.
    
    Pipeline Stages:
    1. Extract: Filter transactions to YTD windows for selected years
    2. Transform: Standardize and combine multi-year datasets
    3. Aggregate: Summarize revenue and quantity by Year, Location, and Item Category
    4. Visualize: Generate interactive Plotly bar charts for comparative analysis
    5. Output: Return structured datasets and pivot tables for downstream use
    """

    # --- Step 1: Filter data to YTD for multiple years ---
    def build_ytd_dataset(df, year, as_of_date=None):
        """
        Filter DataFrame to YTD range for a given year.
        Uses the current date (UTC) unless manually overridden.
        """
        df = df.copy()
        
        if as_of_date is None:
            as_of_date = pd.Timestamp.today(tz='UTC').normalize()
        else:
            # convert user input into a proper timestamp
            as_of_date = pd.to_datetime(as_of_date).tz_localize('UTC').normalize()
        
        start = pd.Timestamp(year, 1, 1, tz='UTC')
        end = pd.Timestamp(year, as_of_date.month, as_of_date.day, tz='UTC')
        
        return df.loc[(df['Paid At'] >= start) & (df['Paid At'] <= end)]

    def aggregate_sales(df, year):
        """Apply YTD filter and tag results with the corresponding year."""
        ytd = build_ytd_dataset(df, year)
        ytd['Year'] = year
        return ytd
        
    # Combine YTD datasets across years for comparison
    df_all_years = pd.concat([
        aggregate_sales(df, y)
        for y in [2026, 2025, 2024, 2023, 2022, 2021]
    ])
    
    df_all_years.to_csv('Output_Files/df_all_years.csv', index=False)

    # --- Step 2: Aggregate sales metrics ---
    sales_summary_df = (
        df_all_years.groupby(['Year', 'Location', 'Item Category'])
        .agg({'Item Qty': 'sum', 'Total': 'sum'})
        .reset_index()
        .sort_values(by=['Year', 'Location', 'Item Category'], ascending=[False, False, True])
    )

    # --- Step 3: Visualization helper (interactive Plotly bar charts) ---
    def plot_pivot(pivot_df, title, ylabel):
        """
        Convert pivot table to long format and render an interactive bar chart.
        Skips plotting if data is empty or non-numeric.
        """
        if pivot_df.empty or pivot_df.select_dtypes(include='number').empty:
            print(f"⚠️ Skipping plot '{title}' — no numeric data to plot")    
            return
           
        # Reshape pivot for Plotly compatibility
        plot_df = pivot_df.reset_index().melt(
            id_vars='Item Category',
            var_name='Year',
            value_name=ylabel
        )

        fig = px.bar(
            plot_df,
            x='Item Category',
            y=ylabel,
            color='Year',
            barmode='group',
            title=title
        )

        fig.update_layout(
            xaxis_title='Item Category',
            yaxis_title=ylabel,
            xaxis_tickangle=45,
            yaxis_tickformat=',.2f'
        )
        
        fig.update_traces(
            hovertemplate='%{x}<br>$%{y:,.2f}<extra></extra>'
        )
        
        fig.show()
    
    # --- Step 4: Generate pivot tables and visualizations ---
    # 4.1 Total sales (both locations combined) ---
    TITLES = {
        "pivot_1": "YTD Sales by Item Category and Year (Locations Combined)",
        "pivot_2": "YTD Sales by Item Category and Year — {}",
        "pivot_3": "YTD Quantities Sold by Item Category and Year (Locations Combined)",
        "pivot_4": "YTD Quantities Sold by Item Category and Year - {}"
    }
    
    pivot_1 = sales_summary_df.pivot_table(
        index='Item Category',
        columns='Year',
        values='Total',
        aggfunc='sum',
        fill_value=0
    ).apply(pd.to_numeric, errors='coerce')
    

    # 4.2 Total sales per location ---
    pivot_2_dict = {}
    for location in sales_summary_df['Location'].unique():
        location_df = sales_summary_df[sales_summary_df['Location'] == location]
        pivot_2 = location_df.pivot_table(
            index='Item Category',
            columns='Year',
            values='Total', 
            aggfunc='sum', 
            fill_value=0
        ).apply(pd.to_numeric, errors='coerce')
        
        # Store each pivot table in a dictionary
        pivot_2_dict[location] = pivot_2

    # 4.3 Quantity sold (both locations combined) ---
    pivot_3 = sales_summary_df.pivot_table(
        index='Item Category', columns='Year', values='Item Qty', aggfunc='sum', fill_value=0
    ).apply(pd.to_numeric, errors='coerce')

    
    # 4.4 Quantities sold per location ---
    pivot_4_dict = {}
    for location in sales_summary_df['Location'].unique():
        location_df = sales_summary_df[sales_summary_df['Location'] == location]
        pivot_4 = location_df.pivot_table(
            index='Item Category', columns='Year', values='Item Qty', aggfunc='sum', fill_value=0
        ).apply(pd.to_numeric, errors='coerce')

        # Store each pivot table in a dictionary
        pivot_4_dict[location] = pivot_4


    # Export Pivot Tables to an Excel file
    with pd.ExcelWriter("Output_Files/YTD_Sales_Report.xlsx") as writer:
        pivot_1.to_excel(writer, sheet_name="YTD_Sales_All_Locations")

        # Pivot 2 (per location)
        for location, df in pivot_2_dict.items():
            sheet_name = f"YTD_Sales_{location}"[:31]
            df.to_excel(writer, sheet_name=sheet_name)

        pivot_3.to_excel(writer, sheet_name="YTD_Qty_All_Locations")

        # Pivot 4 (per location)
        for location, df in pivot_4_dict.items():
            sheet_name = f"YTD_Qty{location}"[:31]
            df.to_excel(writer, sheet_name=sheet_name)
    
    # Print Pivot Table 1 Title, Display Pivot Table 1, Plot Pivot Table 1 Visualization
    print(f"\n {TITLES['pivot_1']}")
    display(pivot_1)
    plot_pivot(pivot_1, TITLES['pivot_1'], 'Total Sales')
    
    for location, df in pivot_2_dict.items():
        title = TITLES['pivot_2'].format(location)
        
        # Print Pivot Table 2 Title, Display Pivot Table 2, Plot Pivot Table 2 Visualization
        print(f"\n {title}")
        display(df)
        plot_pivot(pivot_2, f"YTD Sales by Item Category and Year - {location}", 'Total Sales')

    # Print Pivot Table 3 Title, Display Pivot Table 3, Plot Pivot Table 3 Visualization
    print(f"\n {TITLES['pivot_3']}")
    display(pivot_3)
    plot_pivot(pivot_3, TITLES['pivot_3'], 'Quantity') 

    for location, df in pivot_4_dict.items():
        title = TITLES['pivot_4'].format(location)
        
        # Print Pivot Table 4 Title, Display Pivot Table 4, Plot Pivot Table 4 Visualization
        print(f"\n {title}")
        display(df)
        plot_pivot(pivot_4, f"YTD Qty Sold by Item Category and Year- {location}", 'Quantity')

    # print("✅ Year-To-Date Analysis Pipeline complete.")

    # --- Step 5: Return structured outputs for optional further use ---
    return {
        "pivot_1": pivot_1,
        "pivot_2": pivot_2_dict,
        "pivot_3": pivot_3,
        "pivot_4": pivot_4_dict,
        "sales_summary_df": sales_summary_df,
        "df_all_years": df_all_years
    }

In [3]:
# === WATCHER LOOP ===
FOLDER_PATH = 'Resources'
CHECK_INTERVAL = 30 # time in seconds
processed_files = set() 
master_df = pd.DataFrame()

print(f"Monitoring folder: {FOLDER_PATH}")

cols_to_use = [
      'Name', 'Paid at', 'Fulfilled at', 'Subtotal', 'Shipping', 'Taxes', 'Total',
      'Discount Code', 'Discount Amount', 'Lineitem quantity', 'Lineitem name',
      'Lineitem price', 'Lineitem sku', 'Location', 'Device ID', 'Lineitem discount',
      'Tax 1 Name', 'Tax 1 Value', 'Tax 2 Name', 'Tax 2 Value'
]

col_dtypes = {
    'Name': object,
    'Paid at': object,
    'Fulfilled at': object,
    'Subtotal': 'float64',
    'Shipping': 'float64',
    'Taxes': 'float64',
    'Total': 'float64',
    'Discount Code': object,
    'Discount Amount': 'float64',
    'Lineitem quantity': 'Int64',
    'Lineitem name': object,
    'Lineitem price': 'float64',
    'Lineitem sku': object,
    'Location': object,
    'Device ID': 'float64',
    'Lineitem discount': 'float64',
    'Tax 1 Name': object,
    'Tax 1 Value': 'float64',
    'Tax 2 Name': object,
    'Tax 2 Value': 'float64'
}

while True:
     # Find all CSV files in the folder
    csv_files = [f for f in os.listdir(FOLDER_PATH) if f.endswith(".csv")]

    # Identify new files we haven’t processed yet
    new_files = [f for f in csv_files if f not in processed_files]
    
    if not new_files:
        print("No new CSVs detected. Exiting.")
        break
        
    if new_files:
        print(f"\n📂 New files detected: {new_files}")
        
        new_dataframes = [] # store all newly loaded dataframes
        
        for file in new_files:
            file_path = os.path.join(FOLDER_PATH, file)
            print(f"➡️ Reading {file}...")

            try:
                # Only read the columns we want, with nullable dtypes
                new_df = pd.read_csv(
                    file_path,
                    usecols = lambda x: x.strip() in cols_to_use, # strip whitespace
                     dtype = col_dtypes,
                     low_memory=False,
                     )
                print(f"✅ Successfully loaded: {file} ({new_df.shape[0]} rows)")
                new_dataframes.append(new_df)
                processed_files.add(file)
                
            except Exception as e:
                print(f"⚠️ Failed to load {file}: {e}")
                print("Checking problematic data...")
                temp_df = pd.read_csv(file_path, usecols=lambda x: x.strip() in cols_to_use, dtype=str)
                for col, dtype in col_dtypes.items():
                    if dtype in [float, "float64", "Int64"] and col in temp_df.columns:
                        bad_rows = temp_df[
                        ~temp_df[col]
                        .str.replace(r'[^0-9\.-]', '', regex=True)
                        .str.match(r'^-?\d*\.?\d*$', na=True)
                        ]
                        if not bad_rows.empty:
                            print(f"⚠️ Column '{col}' has non-numeric values:")
                            print(bad_rows[col].unique()[:10])
                
        # ✅ Only proceed if we actually loaded something new
        if new_dataframes:
            combined_new_data = pd.concat(new_dataframes, ignore_index=True)
            print(f"🧩 Combined new data shape: {combined_new_data.shape}")

            # Add to master DataFrame
            master_df = pd.concat([master_df, combined_new_data], ignore_index=True)
            print(f"📊 Master DataFrame now has {master_df.shape[0]} rows.")

            # Run full cleaning/parsing workflow
            master_df = clean_and_parse(master_df)

            # Optional: validate numeric columns
            for col in ["Subtotal", "Shipping", "Taxes", "Total"]:
                bad = master_df[master_df[col].apply(lambda x: isinstance(x, str))]
                if not bad.empty:
                    print(f"⚠️ Non-numeric values found in {col}:")
                    print(bad[col].unique()[:10])

            # ✅ Run visualizations only once, after all new files processed
            print("📈 Preparing visualizations...")
            results = ytd_sales_analysis_pipeline(master_df)
            print("✅ Visualizations complete.")

        else:
            print("No valid new dataframes loaded; skipping analysis.")

    time.sleep(CHECK_INTERVAL)